## inference code for whisper small

In [1]:
import torch
from safetensors import safe_open

# In Whisper, k_proj has no bias across all three attention modules.
# (encoder self-attention, decoder self-attention, and decoder cross-attention).


class WhisperEncoderAttention(torch.nn.Module):
    def __init__(self, config: dict, layer_no: int, safetensor_files: list):
        super().__init__()

        self.d_model = config["d_model"]
        self.num_head = config["encoder_attention_heads"]

        self.files = safetensor_files

        assert self.d_model % self.num_head == 0
        self.head_dim = self.d_model // self.num_head

        self.q_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.k_proj = torch.nn.Linear(self.d_model, self.d_model, bias=False)
        self.v_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.out_proj = torch.nn.Linear(self.d_model, self.d_model)

        # loading pretrained weights
        self.q_proj.weight.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.self_attn.q_proj.weight")
        )
        self.q_proj.bias.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.self_attn.q_proj.bias")
        )
        self.k_proj.weight.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.self_attn.k_proj.weight")
        )
        self.v_proj.weight.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.self_attn.v_proj.weight")
        )
        self.v_proj.bias.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.self_attn.v_proj.bias")
        )
        self.out_proj.weight.data.copy_(
            self.get_tensor(
                f"model.encoder.layers.{layer_no}.self_attn.out_proj.weight"
            )
        )
        self.out_proj.bias.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.self_attn.out_proj.bias")
        )

    @torch.no_grad()
    def forward(self, x):
        B, S, D = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, S, self.num_head, self.head_dim).transpose(1, 2)
        k = k.view(B, S, self.num_head, self.head_dim).transpose(1, 2)
        v = v.view(B, S, self.num_head, self.head_dim).transpose(1, 2)

        attn_score = torch.einsum("bhqd, bhkd -> bhqk", q, k) * (self.head_dim**-0.5)
        wei = torch.nn.functional.softmax(attn_score, dim=-1)

        v_rep = wei @ v
        out = v_rep.transpose(1, 2).contiguous()
        out = out.view(B, S, self.d_model)

        return self.out_proj(out)

    def get_tensor(self, name):
        for f in self.files:
            if name in f.keys():
                return f.get_tensor(name)

In [2]:
class WhisperEncoderFeedForwardNeuralNetwork(torch.nn.Module):

    def __init__(self, config: dict, layer_no: int, safetensor_files: list):
        super().__init__()

        self.d_model = config["d_model"]
        self.d_ff = config["encoder_ffn_dim"]

        self.files = safetensor_files

        self.fc1 = torch.nn.Linear(in_features=self.d_model, out_features=self.d_ff)
        self.fc2 = torch.nn.Linear(in_features=self.d_ff, out_features=self.d_model)

        # loading pretrained weights
        self.fc1.weight.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.fc1.weight")
        )
        self.fc1.bias.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.fc1.bias")
        )
        self.fc2.weight.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.fc2.weight")
        )
        self.fc2.bias.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.fc2.bias")
        )

    @torch.no_grad()
    def forward(self, x):

        output = self.fc1(x)
        output = torch.nn.functional.gelu(output)

        return self.fc2(output)

    def get_tensor(self, name):
        for f in self.files:
            if name in f.keys():
                return f.get_tensor(name)

In [3]:
class WhisperEncoderLayer(torch.nn.Module):

    def __init__(self, config: dict, layer_no, safetensor_files: list):
        super().__init__()

        self.files = safetensor_files

        self.encoder_attn = WhisperEncoderAttention(config, layer_no, self.files)
        self.encoder_ffn = WhisperEncoderFeedForwardNeuralNetwork(
            config, layer_no, self.files
        )

        self.attn_layer_norm = torch.nn.LayerNorm(config["d_model"])
        self.attn_layer_norm.weight.data.copy_(
            self.get_tensor(
                f"model.encoder.layers.{layer_no}.self_attn_layer_norm.weight"
            )
        )
        self.attn_layer_norm.bias.data.copy_(
            self.get_tensor(
                f"model.encoder.layers.{layer_no}.self_attn_layer_norm.bias"
            )
        )

        self.final_layer_norm = torch.nn.LayerNorm(config["d_model"])
        self.final_layer_norm.weight.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.final_layer_norm.weight")
        )
        self.final_layer_norm.bias.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.final_layer_norm.bias")
        )

    @torch.no_grad()
    def forward(self, x):

        residual = x
        x = residual + self.encoder_attn(self.attn_layer_norm(x))

        residual = x
        x = residual + self.encoder_ffn(self.final_layer_norm(x))

        return x

    def get_tensor(self, name):
        for f in self.files:
            if name in f.keys():
                return f.get_tensor(name)

In [4]:
class WhisperEncoder(torch.nn.Module):

    def __init__(self, config: dict, safetensor_files: list):
        super().__init__()

        self.model_config = config
        self.files = safetensor_files

        self.conv1 = torch.nn.Conv1d(
            in_channels=self.model_config["num_mel_bins"],
            out_channels=self.model_config["d_model"],
            kernel_size=3,
            padding=1,
        )
        self.conv1.weight.data.copy_(self.get_tensor("model.encoder.conv1.weight"))
        self.conv1.bias.data.copy_(self.get_tensor("model.encoder.conv1.bias"))

        self.conv2 = torch.nn.Conv1d(
            in_channels=self.model_config["d_model"],
            out_channels=self.model_config["d_model"],
            kernel_size=3,
            stride=2,
            padding=1,
        )
        self.conv2.weight.data.copy_(self.get_tensor("model.encoder.conv2.weight"))
        self.conv2.bias.data.copy_(self.get_tensor("model.encoder.conv2.bias"))

        self.embed_positions = torch.nn.Embedding(
            num_embeddings=self.model_config["max_source_positions"],
            embedding_dim=self.model_config["d_model"],
        )
        self.embed_positions.weight.data.copy_(
            self.get_tensor("model.encoder.embed_positions.weight")
        )

        self.layers = torch.nn.ModuleList(
            [
                WhisperEncoderLayer(self.model_config, i, self.files)
                for i in range(self.model_config["encoder_layers"])
            ]
        )

        self.layer_norm = torch.nn.LayerNorm(config["d_model"])
        self.layer_norm.weight.data.copy_(
            self.get_tensor("model.encoder.layer_norm.weight")
        )
        self.layer_norm.bias.data.copy_(
            self.get_tensor("model.encoder.layer_norm.bias")
        )

    @torch.no_grad()
    def forward(self, x):

        x = torch.nn.functional.gelu(self.conv1(x))
        x = torch.nn.functional.gelu(self.conv2(x))

        x = x.transpose(-2, -1)

        positions = torch.arange(x.size(1), device=x.device)

        x = x + self.embed_positions(positions)

        for layer in self.layers:
            x = layer(x)

        return self.layer_norm(x)

    def get_tensor(self, name):

        for f in self.files:
            if name in f.keys():
                return f.get_tensor(name)

## Decoder

In [5]:
# Only One Seq will be came for process.

class WhisperDecoderMaskedAttention(torch.nn.Module):

    def __init__(self, config: dict, layer_no: int, safetensor_files: list):
        super().__init__()

        self.d_model = config["d_model"]
        self.num_head = config["decoder_attention_heads"]

        self.files = safetensor_files

        assert self.d_model % self.num_head == 0
        self.head_dim = self.d_model // self.num_head

        self.q_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.k_proj = torch.nn.Linear(self.d_model, self.d_model, bias=False)
        self.v_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.out_proj = torch.nn.Linear(self.d_model, self.d_model)

        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(
                    config["max_target_positions"], config["max_target_positions"]
                ),
                diagonal=1,
            ).bool(),
        )

        # weight's
        self.q_proj.weight.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.self_attn.q_proj.weight")
        )
        self.k_proj.weight.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.self_attn.k_proj.weight")
        )
        self.v_proj.weight.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.self_attn.v_proj.weight")
        )
        self.out_proj.weight.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.self_attn.out_proj.weight"
            )
        )

        # bias
        self.q_proj.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.self_attn.q_proj.bias")
        )
        self.v_proj.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.self_attn.v_proj.bias")
        )
        self.out_proj.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.self_attn.out_proj.bias")
        )

    @torch.no_grad()
    def forward(self, x, past_kv: None):

        B, S, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, S, self.num_head, self.head_dim).transpose(1, 2)
        k = k.view(B, S, self.num_head, self.head_dim).transpose(1, 2)
        v = v.view(B, S, self.num_head, self.head_dim).transpose(1, 2)

        if past_kv is not None:
            past_k, past_v = past_kv

            k = torch.cat((past_k, k), dim=2)
            v = torch.cat((past_v, v), dim=2)

        present_kv = (k, v)

        attn_score = (q @ k.transpose(-2, -1)) * (self.head_dim**-0.5)

        if S > 1:
            attn_score = attn_score.masked_fill(self.mask[:S, :S], float("-inf"))

        wei = torch.nn.functional.softmax(attn_score, dim=-1)

        v_rep = wei @ v
        out = v_rep.transpose(1, 2).contiguous()
        out = out.view(B, S, self.d_model)

        out = self.out_proj(out)

        return out, present_kv

    def get_tensor(self, name):

        for f in self.files:
            if name in f.keys():
                return f.get_tensor(name)

In [6]:
class WhisperDecoderCrossAttention(torch.nn.Module):

    def __init__(self, config: dict, layer_no: int, safetensor_files: list):
        super().__init__()

        self.d_model = config["d_model"]
        self.num_head = config["decoder_attention_heads"]

        self.files = safetensor_files

        assert self.d_model % self.num_head == 0
        self.head_dim = self.d_model // self.num_head

        self.q_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.k_proj = torch.nn.Linear(self.d_model, self.d_model, bias=False)
        self.v_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.out_proj = torch.nn.Linear(self.d_model, self.d_model)

        # weight's
        self.q_proj.weight.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.encoder_attn.q_proj.weight"
            )
        )
        self.k_proj.weight.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.encoder_attn.k_proj.weight"
            )
        )
        self.v_proj.weight.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.encoder_attn.v_proj.weight"
            )
        )
        self.out_proj.weight.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.encoder_attn.out_proj.weight"
            )
        )

        # bias
        self.q_proj.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.encoder_attn.q_proj.bias")
        )
        self.v_proj.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.encoder_attn.v_proj.bias")
        )
        self.out_proj.bias.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.encoder_attn.out_proj.bias"
            )
        )

    @torch.no_grad()
    def forward(self, decoder_rep, encoder_rep, past_kv: None):

        DB, DS, _ = decoder_rep.shape
        EB, ES, _ = encoder_rep.shape

        if past_kv is not None:
            k, v = past_kv

        else:
            k = self.k_proj(encoder_rep)
            v = self.v_proj(encoder_rep)

            k = k.view(EB, ES, self.num_head, self.head_dim).transpose(1, 2)
            v = v.view(EB, ES, self.num_head, self.head_dim).transpose(1, 2)

        q = self.q_proj(decoder_rep)
        q = q.view(DB, DS, self.num_head, self.head_dim).transpose(1, 2)

        attn_score = (q @ k.transpose(-2, -1)) * (self.head_dim**-0.5)
        wei = torch.nn.functional.softmax(attn_score, dim=-1)

        v_rep = wei @ v
        out = v_rep.transpose(1, 2).contiguous()
        out = out.view(DB, DS, self.d_model)

        out = self.out_proj(out)

        return out, (k, v)

    def get_tensor(self, name):

        for file in self.files:
            if name in file.keys():
                return file.get_tensor(name)

In [7]:
class WhisperDecoderFeedForwardNeuralNetwork(torch.nn.Module):

    def __init__(self, config: dict, layer_no: int, safetensor_files: list):
        super().__init__()
        self.d_model = config["d_model"]
        self.d_ff = config["decoder_ffn_dim"]

        self.files = safetensor_files

        self.fc1 = torch.nn.Linear(in_features=self.d_model, out_features=self.d_ff)
        self.fc2 = torch.nn.Linear(in_features=self.d_ff, out_features=self.d_model)

        # weight's
        self.fc1.weight.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.fc1.weight")
        )
        self.fc2.weight.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.fc2.weight")
        )

        # bias
        self.fc1.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.fc1.bias")
        )
        self.fc2.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.fc2.bias")
        )

    @torch.no_grad()
    def forward(self, x):

        output = self.fc1(x)
        output = torch.nn.functional.gelu(output)

        return self.fc2(output)

    def get_tensor(self, name):

        for file in self.files:
            if name in file.keys():
                return file.get_tensor(name)

In [8]:
class WhisperDecoderLayer(torch.nn.Module):

    def __init__(self, config: dict, layer_no: int, safetensor_files: list):
        super().__init__()

        self.d_model = config["d_model"]
        self.files = safetensor_files

        self.decoder_mask_attn = WhisperDecoderMaskedAttention(
            config, layer_no, safetensor_files
        )
        self.decoder_cross_attn = WhisperDecoderCrossAttention(
            config, layer_no, safetensor_files
        )
        self.decoder_ffn = WhisperDecoderFeedForwardNeuralNetwork(
            config, layer_no, safetensor_files
        )

        # layer norm
        self.mask_attn_layer_norm = torch.nn.LayerNorm(self.d_model)
        self.cross_attn_layer_norm = torch.nn.LayerNorm(self.d_model)
        self.final_layer_norm = torch.nn.LayerNorm(self.d_model)

        # weight's
        self.mask_attn_layer_norm.weight.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.self_attn_layer_norm.weight"
            )
        )
        self.cross_attn_layer_norm.weight.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.encoder_attn_layer_norm.weight"
            )
        )
        self.final_layer_norm.weight.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.final_layer_norm.weight")
        )

        # bias
        self.mask_attn_layer_norm.bias.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.self_attn_layer_norm.bias"
            )
        )
        self.cross_attn_layer_norm.bias.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.encoder_attn_layer_norm.bias"
            )
        )
        self.final_layer_norm.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.final_layer_norm.bias")
        )

    @torch.no_grad()
    def forward(
        self, from_decoder, from_encoder, masked_attn_kv: None, cross_attn_kv: None
    ):

        residual = from_decoder

        x, present_masked_attn_kv = self.decoder_mask_attn(
            self.mask_attn_layer_norm(residual), masked_attn_kv
        )
        x = residual + x

        residual = x

        x, present_cross_attn_kv = self.decoder_cross_attn(
            self.cross_attn_layer_norm(x), from_encoder, cross_attn_kv
        )
        x = residual + x

        x = x + self.decoder_ffn(self.final_layer_norm(x))

        return x, present_masked_attn_kv, present_cross_attn_kv

    def get_tensor(self, name):
        for file in self.files:
            if name in file.keys():
                return file.get_tensor(name)

In [9]:
class WhisperDecoder(torch.nn.Module):

    def __init__(self, config: dict, safetensor_files: list):
        super().__init__()

        self.model_config = config
        self.files = safetensor_files

        self.embed_tokens = torch.nn.Embedding(
            num_embeddings=self.model_config["vocab_size"],
            embedding_dim=self.model_config["d_model"],
            padding_idx=self.model_config["pad_token_id"],
        )
        self.embed_positions = torch.nn.Embedding(
            num_embeddings=self.model_config["max_target_positions"],
            embedding_dim=self.model_config["d_model"],
        )
        self.layers = torch.nn.ModuleList(
            [
                WhisperDecoderLayer(self.model_config, i, self.files)
                for i in range(self.model_config["decoder_layers"])
            ]
        )
        self.layer_norm = torch.nn.LayerNorm(self.model_config["d_model"])

        # weight's
        self.embed_tokens.weight.data.copy_(
            self.get_tensor("model.decoder.embed_tokens.weight")
        )
        self.embed_positions.weight.data.copy_(
            self.get_tensor("model.decoder.embed_positions.weight")
        )
        self.layer_norm.weight.data.copy_(
            self.get_tensor("model.decoder.layer_norm.weight")
        )
        # bias
        self.layer_norm.bias.data.copy_(
            self.get_tensor("model.decoder.layer_norm.bias")
        )
        # kv cache
        self.masked_attn_kv = [None] * self.model_config["decoder_layers"]
        self.cross_attn_kv = [None] * self.model_config["decoder_layers"]

        self.current_pos = None

    @torch.no_grad()
    def forward(self, from_decoder, from_encoder):

        x = self.embed_tokens(from_decoder)

        if self.current_pos is None:
            self.current_pos = x.size(1)
            positions = torch.arange(self.current_pos, device=x.device)
            x = x + self.embed_positions(positions)

        else:
            position = torch.tensor([self.current_pos], device=x.device)
            x =  x + self.embed_positions(position)
            self.current_pos += 1

        for i, layer in enumerate(self.layers):
            x, self.masked_attn_kv[i], self.cross_attn_kv[i] = layer(
                x, from_encoder, self.masked_attn_kv[i], self.cross_attn_kv[i]
            )

        return self.layer_norm(x)

    def get_tensor(self, name):
        for file in self.files:
            if name in file.keys():
                return file.get_tensor(name)

    def clean_cache(self):
        self.masked_attn_kv = [None] * self.model_config["decoder_layers"]
        self.cross_attn_kv = [None] * self.model_config["decoder_layers"]
        self.current_pos = None

In [10]:
# Implementing the same audio loading, trimming, and padding logic used in the original Whisper library.

import os
from math import gcd
from functools import lru_cache
from typing import Optional, Union
import logging

logger = logging.getLogger(__name__)


import numpy as np
from scipy.signal import resample_poly
import soundfile as sf
import torch
import torch.nn.functional as F
from tokenizers import Tokenizer

# hard-coded audio hyperparameters
SAMPLE_RATE = 16000
N_FFT = 400
HOP_LENGTH = 160
CHUNK_LENGTH = 30
N_SAMPLES = CHUNK_LENGTH * SAMPLE_RATE  # 480000 samples in a 30-second chunk
N_FRAMES = N_SAMPLES // HOP_LENGTH  # 3000 frames in a mel spectrogram input

N_SAMPLES_PER_TOKEN = HOP_LENGTH * 2  # the initial convolutions has stride 2
FRAMES_PER_SECOND = SAMPLE_RATE // HOP_LENGTH  # 10ms per audio frame
TOKENS_PER_SECOND = SAMPLE_RATE // N_SAMPLES_PER_TOKEN  # 20ms per audio token


def pad_or_trim(array, length: int = N_SAMPLES, *, axis: int = -1):
    """
    Pad or trim the audio array to N_SAMPLES, as expected by the encoder.
    """
    if torch.is_tensor(array):
        if array.shape[axis] > length:
            array = array.index_select(
                dim=axis, index=torch.arange(length, device=array.device)
            )

        if array.shape[axis] < length:
            pad_widths = [(0, 0)] * array.ndim
            pad_widths[axis] = (0, length - array.shape[axis])
            array = F.pad(array, [pad for sizes in pad_widths[::-1] for pad in sizes])
    else:
        if array.shape[axis] > length:
            array = array.take(indices=range(length), axis=axis)

        if array.shape[axis] < length:
            pad_widths = [(0, 0)] * array.ndim
            pad_widths[axis] = (0, length - array.shape[axis])
            array = np.pad(array, pad_widths)

    return array


@lru_cache(maxsize=None)
def mel_filters(device, n_mels: int) -> torch.Tensor:
    """
    load the mel filterbank matrix for projecting STFT into a Mel spectrogram.
    Allows decoupling librosa dependency; saved using:

        np.savez_compressed(
            "mel_filters.npz",
            mel_80=librosa.filters.mel(sr=16000, n_fft=400, n_mels=80),
            mel_128=librosa.filters.mel(sr=16000, n_fft=400, n_mels=128),
        )
    """
    assert n_mels in {80, 128}, f"Unsupported n_mels: {n_mels}"

    # This may cause build issues because the file is not included in the build.
    filters_path = (
        "/home/vishwjeet/Desktop/Projects/LinguSync/backend/models/stt/mel_filters.npz"
    )

    with np.load(filters_path, allow_pickle=False) as f:
        return torch.from_numpy(f[f"mel_{n_mels}"]).to(device)


def log_mel_spectrogram(
    audio: Union[np.ndarray, torch.Tensor],
    n_mels: int = 80,
    padding: int = 0,
    device: Optional[Union[str, torch.device]] = None,
):
    """
    Compute the log-Mel spectrogram of

    Parameters
    ----------
    audio: Union[np.ndarray, torch.Tensor], shape = (*)
        NumPy array or Tensor containing the audio waveform in 16 kHz

    n_mels: int
        The number of Mel-frequency filters, only 80 and 128 are supported

    padding: int
        Number of zero samples to pad to the right

    device: Optional[Union[str, torch.device]]
        If given, the audio tensor is moved to this device before STFT

    Returns
    -------
    torch.Tensor, shape = (1, n_mels, n_frames)
        A Tensor that contains the Mel spectrogram
    """
    if not torch.is_tensor(audio):
        audio = torch.from_numpy(audio)

    if device is not None:
        audio = audio.to(device)
    if padding > 0:
        audio = F.pad(audio, (0, padding))
    window = torch.hann_window(N_FFT).to(audio.device)
    stft = torch.stft(audio, N_FFT, HOP_LENGTH, window=window, return_complex=True)
    magnitudes = stft[..., :-1].abs() ** 2

    filters = mel_filters(audio.device, n_mels)
    mel_spec = filters @ magnitudes

    log_spec = torch.clamp(mel_spec, min=1e-10).log10()
    log_spec = torch.maximum(log_spec, log_spec.max() - 8.0)
    log_spec = (log_spec + 4.0) / 4.0

    return log_spec.unsqueeze(0)


def split_audio(file: str, sr: int = 16000, chunk_duration: int = 30) -> list:
    """
    Split the input audio file into fixed-duration chunks.

    Parameters
    ----------
    file : str
        Path to the input audio file.

    sr : int, default=16000
        Sample rate of the output audio in Hz.

    chunk_duration : int, default=30
        Duration of each audio chunk in seconds.

    Returns
    -------
    list[np.ndarray]
        A list of audio chunks represented as NumPy arrays.
        Each chunk contains mono, float32 audio at the specified sample rate.
    """

    # Load audio
    audio, original_sr = sf.read(file, dtype="float32")

    # Convert stereo to mono
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    # Resample if necessary
    if original_sr != sr:
        divisor = gcd(original_sr, sr)
        audio = resample_poly(audio, sr // divisor, original_sr // divisor).astype(
            np.float32
        )

    # Split into chunks
    chunk_size = chunk_duration * sr

    chunks = [
        audio[start : start + chunk_size] for start in range(0, len(audio), chunk_size)
    ]

    return chunks


def save_transcript(
    script: str = None, file_name: str = "transcript.txt", save_dir: str = None
) -> bool:
    """
    Save the speech-to-text model's generated transcript (output) to a text file.

    Parameters
    ----------
    script : str, default=None
        Transcript or translation generated by the STT model.

    file_name : str, default="transcript.txt"
        Name of the file to save the transcript as.

    save_dir : str, default=None
        Path to the directory where the file should be saved.

    Returns
    -------
    str
        Path to the saved transcript file.
    """

    try:

        if not save_dir or not os.path.isdir(save_dir):
            raise FileNotFoundError(f"Directory not found: {save_dir}")

        file_path = os.path.join(save_dir, file_name)

        with open(file_path, "w", encoding="utf-8") as f:
            f.write(script)

        return file_path

    except Exception as E:
        logger.error(E)
        return None


In [11]:
class WhisperSmall(torch.nn.Module):
    """
    Whisper Small is a lightweight speech-to-text (STT) model with approximately
    244 million parameters. It officially supports mono audio clips of up to
    30 seconds in duration.

    Parameters
    ----------
    config : dict
        Official configuration dictionary of the model.

    tokenizer : str
        Path to the directory containing the model's tokenizer files.

    safetensor_files : list
        List of paths to the model's weight files in SafeTensors format.

    Returns
    -------
    str
        Transcription of the input audio as a string.
    """

    def __init__(self, config: dict, tokenizer: str, safetensor_files: list[str]):
        super().__init__()

        self.model_config = config
        self.tk = Tokenizer.from_file(tokenizer)

        self.bos_token_id = config["bos_token_id"]
        self.decoder_start_token_id = config["decoder_start_token_id"]
        self.eos_token_id = config["eos_token_id"]
        self.max_length = config["max_length"]

        # Whisper timestamp tokens
        self.timestamp_begin = 50364
        self.timestamp_rate = 0.02
        self.no_timestamps_token_id = 50363

        self.max_initial_timestamp = 1.0
        self.max_initial_timestamp_index = int(
            self.max_initial_timestamp / self.timestamp_rate
        )

        self.files = [
            safe_open(file, framework="pt", device="cpu") for file in safetensor_files
        ]

        self.encoder = WhisperEncoder(self.model_config, self.files)
        self.decoder = WhisperDecoder(self.model_config, self.files)

        # Tied output projection
        self.back_proj = torch.nn.Embedding(
            num_embeddings=self.model_config["vocab_size"],
            embedding_dim=self.model_config["d_model"],
        )

        self.back_proj.weight.data.copy_(
            self.get_tensor("model.decoder.embed_tokens.weight")
        )

    def is_timestamp_token(self, token_id):
        return token_id >= self.timestamp_begin

    def token_to_timestamp(self, token_id):
        return (token_id - self.timestamp_begin) * self.timestamp_rate

    def apply_timestamp_rules(self, logits, generated_tokens):

        logits = logits[:, -1, :]

        # Suppress <|notimestamps|>
        logits[:, self.no_timestamps_token_id] = -float("inf")

        # Timestamp rules
        if len(generated_tokens) > 0:

            last_was_timestamp = (
                len(generated_tokens) >= 1
                and generated_tokens[-1] >= self.timestamp_begin
            )

            penultimate_was_timestamp = (
                len(generated_tokens) < 2
                or generated_tokens[-2] >= self.timestamp_begin
            )

            # Timestamps have to appear in pairs
            if last_was_timestamp:

                if penultimate_was_timestamp:
                    # Two consecutive timestamps:
                    # next token must be normal text/EOT.
                    logits[:, self.timestamp_begin :] = -float("inf")

                else:
                    # One timestamp:
                    # next token must be another timestamp.
                    logits[:, : self.eos_token_id] = -float("inf")

            timestamps = torch.tensor(
                [token for token in generated_tokens if token >= self.timestamp_begin],
                device=logits.device,
            )

            if timestamps.numel() > 0:

                # Timestamps cannot decrease.
                #
                # If we just generated the first timestamp of a pair,
                # the second timestamp must be strictly greater.
                if last_was_timestamp and not penultimate_was_timestamp:

                    timestamp_last = timestamps[-1]

                else:

                    timestamp_last = timestamps[-1] + 1

                logits[:, self.timestamp_begin : timestamp_last] = -float("inf")

        # First generated token must be a timestamp
        if len(generated_tokens) == 0:

            logits[:, : self.timestamp_begin] = -float("inf")

            # Maximum initial timestamp = 1 second
            last_allowed = self.timestamp_begin + self.max_initial_timestamp_index

            logits[:, last_allowed + 1 :] = -float("inf")

        # If the combined probability of all timestamp tokens is
        # greater than the probability of the best text token,
        # force a timestamp.

        logprobs = torch.nn.functional.log_softmax(
            logits.float(),
            dim=-1,
        )

        timestamp_logprob = torch.logsumexp(
            logprobs[:, self.timestamp_begin :],
            dim=-1,
        )

        max_text_token_logprob = torch.max(
            logprobs[:, : self.timestamp_begin],
            dim=-1,
        ).values

        if timestamp_logprob[0] > max_text_token_logprob[0]:

            logits[:, : self.timestamp_begin] = -float("inf")

        return logits

    @torch.no_grad()
    def forward(
        self,
        audio_path: str,
        language_token: str = "<|hi|>",
        task_id: int = 50359,
        return_timestamps: bool = False,
        device: torch.device = torch.device("cpu"),
    ) -> str:
        """
        Transcribe a single audio file at a time.

        NOTE: The model now supports audio inputs longer than 30 seconds.
            However, timestamp generation has not yet been implemented,
            causing timestamps to restart after every 30 seconds.

        Parameters
        ----------
        audio_file : str
            Path to the input audio file.

        language_token : str
            Language code specifying the language of the audio.
            Refer to the official Hugging Face model repository for
            the complete list of supported language tokens.

            Example: <|en|> : 50259

        task : str
            Specifies the transcription task:
            - "transcribe": Transcribe the audio in its original language.
            - "translate": Translate the audio into English.

            <|transcribe|> : 50359
            <|translate|>  : 50358

        device : str, default="cpu"
            Device on which the model will perform inference.
            Use "cuda" for GPU acceleration or "cpu" for CPU processing.

        return_timestamps : bool, default=False
            Whether to include timestamps in the transcription output.
        """

        transcribe_tokens = []
        lang_id = self.tk.token_to_id(language_token)

        if not lang_id:
            raise ValueError(f"Invalid Language token or Id Token: {language_token}, Id:{lang_id}")

        for chunk in split_audio(audio_path):

            generated_tokens = []

            x = log_mel_spectrogram(pad_or_trim(chunk), device=device)

            encoder_output = self.encoder(x)

            prompt = [
                self.decoder_start_token_id,
                lang_id,
                task_id,
            ]

            if not return_timestamps:
                prompt.append(self.no_timestamps_token_id)

            decoder_input = torch.tensor([prompt], device=x.device)

            self.decoder.clean_cache()

            for _ in range(self.max_length):

                decoder_output = self.decoder(decoder_input, encoder_output)

                logits = decoder_output @ self.back_proj.weight.T

                if return_timestamps:
                    next_logits = self.apply_timestamp_rules(logits, generated_tokens)
                else:
                    next_logits = logits[:, -1, :].clone()

                    # Prevent timestamp generation
                    next_logits[:, self.timestamp_begin :] = -float("inf")

                predicted_token = torch.argmax(next_logits, dim=-1).item()

                if predicted_token == self.eos_token_id:
                    break

                generated_tokens.append(predicted_token)
                transcribe_tokens.append(predicted_token)

                decoder_input = torch.tensor(
                    [[predicted_token]],
                    device=decoder_output.device,
                )

        return self.tk.decode(transcribe_tokens)

    def get_tensor(self, name):

        for file in self.files:
            if name in file.keys():
                return file.get_tensor(name)

        raise KeyError(f"Tensor not found: {name}")

In [12]:
import json

config_file = json.load(open("config.json"))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = WhisperSmall(
    config_file,
    "tokenizer.json",
    ["model.safetensors"]
    ).to(device)

In [ ]:
input_file = "/home/vishwjeet/Downloads/generated_audio.wav"
model_output = model(input_file, device=device)
model_output

In [37]:
new_file_chunks = split_audio(input_file)
default_chunk = split_audio("/home/vishwjeet/Downloads/with_noise_hindi_clip.mp3")[0]

In [38]:
print("Before Padding size of chunk one: ", new_file_chunks[0].shape)
print("After padding size of chunk one: ", pad_or_trim(new_file_chunks[0]).shape)
print("Spectrogram of chunk one: ", log_mel_spectrogram(pad_or_trim(new_file_chunks[0])).shape)

Before Padding size of chunk one:  (480000,)
After padding size of chunk one:  (480000,)
Spectrogram of chunk one:  torch.Size([1, 80, 3000])


In [39]:
print("Before Padding size of chunk Two: ", new_file_chunks[1].shape)
print("After padding size of chunk Two: ", pad_or_trim(new_file_chunks[1]).shape)
print("Spectrogram of chunk Two: ", log_mel_spectrogram(pad_or_trim(new_file_chunks[1])).shape)

Before Padding size of chunk Two:  (159659,)
After padding size of chunk Two:  (480000,)
Spectrogram of chunk Two:  torch.Size([1, 80, 3000])


In [36]:
print("Before Padding size of chunk Default: ", default_chunk.shape)
print("After padding size of chunk Default: ", pad_or_trim(default_chunk).shape)
print("Spectrogram of chunk Default: ", log_mel_spectrogram(pad_or_trim(default_chunk)).shape)

Before Padding size of chunk Default:  (128918,)
After padding size of chunk Default:  (480000,)
Spectrogram of chunk Default:  torch.Size([1, 80, 3000])
